### Engenharia de features (feature enginnering)
O objetivo desta etapa é utilizar as bases de dados elencadas na Análise Exploratória como sendo as mais pertinentes para treinar um modelo que prevê a probabilidade de alfabetização, e utiliza-las para gerar as features que irão servir como base de treino e teste dos modelos.

Neste notebook, será feita:
1. A importação das bases de dados em formato .csv
2. A seleção das colunas (features) pertinentes, já elencadas na EDA
3. O tratamento destas colunas
4. A junção das bases de dados utilizando o ano e o ID do município como chaves
5. A exportação das features em formato .csv

Aqui, não serão abordadas a divisão treino/teste nem a normalização dos dados, pois estes processos são particulares para cada modelo, e o objetivo do trabalho é testar mais de um modelo de aprendizado de máquina diferente. Portanto, estes processos serão descritos nos scripts treinamento dos modelos.

### Importação das bibliotecas

In [63]:
import pandas as pd
from pathlib import Path

base_path = Path.cwd().parent / 'data'

### Base de dados - Indicadores municipais de alfabetização
Esta tabela será a "base" do dataset de treinamento do modelo, pois o mesmo contém as informações mais importantes para tal, além da variável target do modelo, que é a coluna `pc_indicador_alfabetizacao`, que indica o quantos % dos alunos daquela observação são alfabetizados.

A tabela está agrupada pelas colunas `ano`, `id_municipio` e `ds_rede` (indica se está sendo observada a rede Privada, Municipal ou Estadual), e contém dados de 2023 e 2024.

In [64]:
wide = pd.read_csv(base_path / 'wide_analitica_alfabetizacao.csv')

wide['id_municipio'] = wide['id_municipio'] // 10 # Ajuste de formato de chave pra fazer o join com o codigo_ibge
wide = wide.rename(columns={"nu_ano":"ano"})        
wide = wide[['ano','id_municipio','nome_uf','nome_regiao','pc_indicador_alfabetizacao', 'vl_proficiencia_media',
            'vl_proficiencia_mediana','qt_alunos_avaliados','nu_serie','ds_rede']] # Manter apenas as colunas selecionadas na EDA
wide['ano'] = wide['ano'].astype(str)
# Ajuste da escala de colunas percentuais
wide['pc_indicador_alfabetizacao'] = round(wide['pc_indicador_alfabetizacao']/100, 4)
wide.head(5)

,ano,id_municipio,nome_uf,nome_regiao,pc_indicador_alfabetizacao,vl_proficiencia_media,vl_proficiencia_mediana,qt_alunos_avaliados,nu_serie,ds_rede
0,2023,310180,Minas Gerais,Sudeste,0.4898,737.60,734.01,49,2,Municipal
1,2023,310260,Minas Gerais,Sudeste,0.5718,747.21,751.72,376,2,Municipal
2,2023,310410,Minas Gerais,Sudeste,0.8706,791.68,795.49,85,2,Municipal
3,2023,310740,Minas Gerais,Sudeste,0.4440,736.56,737.62,268,2,Municipal
4,2023,310940,Minas Gerais,Sudeste,0.3791,730.09,728.07,277,2,Municipal


In [65]:
wide.info()

<class 'pandas.DataFrame'>
RangeIndex: 12417 entries, 0 to 12416
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ano                         12417 non-null  str    
 1   id_municipio                12417 non-null  int64  
 2   nome_uf                     12417 non-null  str    
 3   nome_regiao                 12417 non-null  str    
 4   pc_indicador_alfabetizacao  12417 non-null  float64
 5   vl_proficiencia_media       12417 non-null  float64
 6   vl_proficiencia_mediana     12417 non-null  float64
 7   qt_alunos_avaliados         12417 non-null  int64  
 8   nu_serie                    12417 non-null  int64  
 9   ds_rede                     12417 non-null  str    
dtypes: float64(3), int64(3), str(4)
memory usage: 970.2 KB


In [66]:
wide.duplicated(['id_municipio','ano','ds_rede']).sum() # Verificação de unicidade

np.int64(0)

### Base de dados - Cadastro Único

Tabela que contém dados, agrupados por mês/ano e município (representado pela coluna `codigo_ibge`), do Cadastro Único, será utilizada para enriquecer o dataset com dados geográficos dos municípios analisados. Para realizar a junção com a tabela `wide_analitica_alfabetizacao`, serão filtradas apenas as observações do mês de dezembro de cada ano, que representa a última aferição das métricas realizada naquele ano em específico. Este processo é necessário dada a natureza dos indicadores, que representam não um dado cumulativo, mas sim um "estoque", ou seja, o total daquela métrica naquele momento.

In [67]:
cad = pd.read_csv(base_path / 'dados_cadunico_2023_a_2026.csv')
cad['mes'] = cad['ano_mes'].str.split('-').str[1]
cad['ano'] = cad['ano_mes'].str.split('-').str[0]
cad = cad[cad['ano'].isin(['2023','2024'])] # Manter apenas os dados de 2023 e 2024, mantendo o mesmo range de data da wide_analitica_alfabetizacao
cad = cad[cad['mes'] == '12'] # Já que os contadores são atualizados de mês a mês, mantém apenas o contador do último mês do ano
cad['taxa_atualizacao_geral_pct'] = round(cad['taxa_atualizacao_geral_pct']/100, 4)
cad['taxa_atualizacao_ate_meio_sm_pct'] = round(cad['taxa_atualizacao_ate_meio_sm_pct']/100, 4)
cad = cad[['codigo_ibge', 'ano', 'qtd_fam_pobreza','qtd_fam_baixa_renda','qtd_fam_ate_meio_sm',
            'qtd_fam_renda_zero','taxa_atualizacao_geral_pct','taxa_atualizacao_ate_meio_sm_pct']] # Manter apenas as colunas selecionadas na EDA
cad.head(5)

,codigo_ibge,ano,qtd_fam_pobreza,qtd_fam_baixa_renda,qtd_fam_ate_meio_sm,qtd_fam_renda_zero,taxa_atualizacao_geral_pct,taxa_atualizacao_ate_meio_sm_pct
61270,110001,2023,1230,1026,2256,34,0.6888,0.7752
61271,110002,2023,5650,3954,9604,953,0.7234,0.8136
61272,110003,2023,213,259,472,14,0.7585,0.7986
61273,110004,2023,4971,3436,8407,1279,0.7326,0.8291
61274,110005,2023,931,536,1467,17,0.6696,0.7668


In [68]:
cad.info()

<class 'pandas.DataFrame'>
Index: 11140 entries, 61270 to 133679
Data columns (total 8 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   codigo_ibge                       11140 non-null  int64  
 1   ano                               11140 non-null  object 
 2   qtd_fam_pobreza                   11140 non-null  int64  
 3   qtd_fam_baixa_renda               11140 non-null  int64  
 4   qtd_fam_ate_meio_sm               11140 non-null  int64  
 5   qtd_fam_renda_zero                11140 non-null  int64  
 6   taxa_atualizacao_geral_pct        11140 non-null  float64
 7   taxa_atualizacao_ate_meio_sm_pct  11140 non-null  float64
dtypes: float64(2), int64(5), object(1)
memory usage: 783.3+ KB


In [69]:
cad.duplicated(['codigo_ibge','ano']).sum() # Verificação de unicidade

np.int64(0)

### Base de dados - Censo escolar

Tabela que contém dados oriundos do censo escolar, com dados relacionados à situação das escolas brasileiras. Por padrão, esta tabela está agrupada por escola, portanto, foi realizado o tratamento para que fique agrupado por ano e município, e as métricas foram ajustadas para tal.

In [70]:
censo = pd.read_csv(base_path / 'censo_escolar_2023_2024.csv')
censo['id_municipio'] = censo['id_municipio'] // 10 # Ajuste de formato de chave pra fazer o join com o codigo_ibge
censo['ano'] = censo['ano'].astype(str)
censo = censo.groupby(['ano', 'id_municipio']).agg(
    qtd_escolas=('id_escola', 'count'),
    pct_urbana=('tipo_localizacao', lambda x: (x == 1).mean() * 100),
    pct_rural=('tipo_localizacao', lambda x: (x == 2).mean() * 100)
).reset_index() # Agrupamento da base por ano e município para manter na mesma granularidade das outras bases
censo['pct_urbana'] = round(censo['pct_urbana']/100, 4)
censo['pct_rural'] = round(censo['pct_rural']/100, 4)
censo.head()

,ano,id_municipio,qtd_escolas,pct_urbana,pct_rural
0,2023,110001,31,0.5161,0.4839
1,2023,110002,55,0.7818,0.2182
2,2023,110003,9,0.7778,0.2222
3,2023,110004,65,0.7077,0.2923
4,2023,110005,15,0.9333,0.0667


In [71]:
censo.info()

<class 'pandas.DataFrame'>
RangeIndex: 11140 entries, 0 to 11139
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ano           11140 non-null  str    
 1   id_municipio  11140 non-null  int64  
 2   qtd_escolas   11140 non-null  int64  
 3   pct_urbana    11140 non-null  float64
 4   pct_rural     11140 non-null  float64
dtypes: float64(2), int64(2), str(1)
memory usage: 435.3 KB


In [72]:
censo.duplicated(['id_municipio','ano']).sum() # Verificação de unicidade

np.int64(0)

### Junção das bases de dados - 

Como mencionado anteriormente, as tabelas do cadastro único e do censo serão unidas na tabela de dados de alfabetização utilizando o código da cidade e o ano como chaves

In [73]:
merged_wide_cad = pd.merge(wide, cad, left_on=['id_municipio','ano'], right_on=['codigo_ibge','ano'], how='left')
merged_wide_cad_censo = pd.merge(merged_wide_cad, censo, on=['id_municipio','ano'], how='left')
merged_wide_cad_censo = merged_wide_cad_censo.drop(columns=['codigo_ibge'])
merged_wide_cad_censo.head()

,ano,id_municipio,nome_uf,nome_regiao,pc_indicador_alfabetizacao,vl_proficiencia_media,vl_proficiencia_mediana,qt_alunos_avaliados,nu_serie,ds_rede,qtd_fam_pobreza,qtd_fam_baixa_renda,qtd_fam_ate_meio_sm,qtd_fam_renda_zero,taxa_atualizacao_geral_pct,taxa_atualizacao_ate_meio_sm_pct,qtd_escolas,pct_urbana,pct_rural
0,2023,310180,Minas Gerais,Sudeste,0.4898,737.60,734.01,49,2,Municipal,1032,307,1339,560,0.7553,0.8017,11,0.7273,0.2727
1,2023,310260,Minas Gerais,Sudeste,0.5718,747.21,751.72,376,2,Municipal,1343,987,2330,487,0.7099,0.8045,35,0.7714,0.2286
2,2023,310410,Minas Gerais,Sudeste,0.8706,791.68,795.49,85,2,Municipal,288,211,499,151,0.5359,0.7748,5,1.0000,0.0000
3,2023,310740,Minas Gerais,Sudeste,0.4440,736.56,737.62,268,2,Municipal,3194,1618,4812,2053,0.7108,0.8459,38,0.9737,0.0263
4,2023,310940,Minas Gerais,Sudeste,0.3791,730.09,728.07,277,2,Municipal,4477,924,5401,93,0.8051,0.9036,81,0.3457,0.6543


### Codificação das variáveis categóricas

In [74]:
merged_wide_cad_censo.dtypes # Análise de quais variáveis precisam passar pelo tratamento

ano                                  object
id_municipio                          int64
nome_uf                                 str
nome_regiao                             str
pc_indicador_alfabetizacao          float64
vl_proficiencia_media               float64
vl_proficiencia_mediana             float64
qt_alunos_avaliados                   int64
nu_serie                              int64
ds_rede                                 str
qtd_fam_pobreza                       int64
qtd_fam_baixa_renda                   int64
qtd_fam_ate_meio_sm                   int64
qtd_fam_renda_zero                    int64
taxa_atualizacao_geral_pct          float64
taxa_atualizacao_ate_meio_sm_pct    float64
qtd_escolas                           int64
pct_urbana                          float64
pct_rural                           float64
dtype: object

### Colunas `ano`, `ds_rede` e `nome_regiao`

Para a codificação destas colunas, será utilizado o Dummy Encoding por dois motivos:
1. São colunas que possuem baixa cardinalidade (ou seja, poucas categorias), ou seja, serão criadas poucas colunas adicionais
2. O Dummy Encoding é compatível com praticamente qualquer modelo

In [75]:
merged_wide_cad_censo = pd.get_dummies(merged_wide_cad_censo, columns=['ano', 'ds_rede','nome_regiao'], dtype=int, drop_first=True)
merged_wide_cad_censo.columns = merged_wide_cad_censo.columns.str.lower()
merged_wide_cad_censo.head()

,id_municipio,nome_uf,pc_indicador_alfabetizacao,vl_proficiencia_media,vl_proficiencia_mediana,qt_alunos_avaliados,nu_serie,qtd_fam_pobreza,qtd_fam_baixa_renda,qtd_fam_ate_meio_sm,...,qtd_escolas,pct_urbana,pct_rural,ano_2024,ds_rede_municipal,ds_rede_privada,nome_regiao_nordeste,nome_regiao_norte,nome_regiao_sudeste,nome_regiao_sul
0,310180,Minas Gerais,0.4898,737.60,734.01,49,2,1032,307,1339,...,11,0.7273,0.2727,0,1,0,0,0,1,0
1,310260,Minas Gerais,0.5718,747.21,751.72,376,2,1343,987,2330,...,35,0.7714,0.2286,0,1,0,0,0,1,0
2,310410,Minas Gerais,0.8706,791.68,795.49,85,2,288,211,499,...,5,1.0000,0.0000,0,1,0,0,0,1,0
3,310740,Minas Gerais,0.4440,736.56,737.62,268,2,3194,1618,4812,...,38,0.9737,0.0263,0,1,0,0,0,1,0
4,310940,Minas Gerais,0.3791,730.09,728.07,277,2,4477,924,5401,...,81,0.3457,0.6543,0,1,0,0,0,1,0


### Coluna `nome_uf`

A coluna de UF (unidade federativa, ou estado), por sua vez, será categorizada utilizando o Frequency Encoding, dado que a mesma possui uma cardinalidade maior, e utilizar o dummy encoding criaria um número excessivo de colunas.

In [76]:
freq = merged_wide_cad_censo['nome_uf'].value_counts(normalize=True)
merged_wide_cad_censo['uf_freq'] = merged_wide_cad_censo['nome_uf'].map(freq)
merged_wide_cad_censo = merged_wide_cad_censo.drop(columns=['nome_uf'])
merged_wide_cad_censo.head()

,id_municipio,pc_indicador_alfabetizacao,vl_proficiencia_media,vl_proficiencia_mediana,qt_alunos_avaliados,nu_serie,qtd_fam_pobreza,qtd_fam_baixa_renda,qtd_fam_ate_meio_sm,qtd_fam_renda_zero,...,pct_urbana,pct_rural,ano_2024,ds_rede_municipal,ds_rede_privada,nome_regiao_nordeste,nome_regiao_norte,nome_regiao_sudeste,nome_regiao_sul,uf_freq
0,310180,0.4898,737.60,734.01,49,2,1032,307,1339,560,...,0.7273,0.2727,0,1,0,0,0,1,0,0.175485
1,310260,0.5718,747.21,751.72,376,2,1343,987,2330,487,...,0.7714,0.2286,0,1,0,0,0,1,0,0.175485
2,310410,0.8706,791.68,795.49,85,2,288,211,499,151,...,1.0000,0.0000,0,1,0,0,0,1,0,0.175485
3,310740,0.4440,736.56,737.62,268,2,3194,1618,4812,2053,...,0.9737,0.0263,0,1,0,0,0,1,0,0.175485
4,310940,0.3791,730.09,728.07,277,2,4477,924,5401,93,...,0.3457,0.6543,0,1,0,0,0,1,0,0.175485


### Tabela final

A última alteração antes da exportação das features é a remoção da coluna `id_municipio`, pois utiliza-la no treinamento do modelo pode fazer com que o mesmo aprenda padrões espúrios a partir do código do IBGE.

In [77]:
merged_wide_cad_censo = merged_wide_cad_censo.drop(columns=['id_municipio'])
merged_wide_cad_censo.head()

,pc_indicador_alfabetizacao,vl_proficiencia_media,vl_proficiencia_mediana,qt_alunos_avaliados,nu_serie,qtd_fam_pobreza,qtd_fam_baixa_renda,qtd_fam_ate_meio_sm,qtd_fam_renda_zero,taxa_atualizacao_geral_pct,...,pct_urbana,pct_rural,ano_2024,ds_rede_municipal,ds_rede_privada,nome_regiao_nordeste,nome_regiao_norte,nome_regiao_sudeste,nome_regiao_sul,uf_freq
0,0.4898,737.60,734.01,49,2,1032,307,1339,560,0.7553,...,0.7273,0.2727,0,1,0,0,0,1,0,0.175485
1,0.5718,747.21,751.72,376,2,1343,987,2330,487,0.7099,...,0.7714,0.2286,0,1,0,0,0,1,0,0.175485
2,0.8706,791.68,795.49,85,2,288,211,499,151,0.5359,...,1.0000,0.0000,0,1,0,0,0,1,0,0.175485
3,0.4440,736.56,737.62,268,2,3194,1618,4812,2053,0.7108,...,0.9737,0.0263,0,1,0,0,0,1,0,0.175485
4,0.3791,730.09,728.07,277,2,4477,924,5401,93,0.8051,...,0.3457,0.6543,0,1,0,0,0,1,0,0.175485


In [78]:
merged_wide_cad_censo.info()

<class 'pandas.DataFrame'>
RangeIndex: 12417 entries, 0 to 12416
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   pc_indicador_alfabetizacao        12417 non-null  float64
 1   vl_proficiencia_media             12417 non-null  float64
 2   vl_proficiencia_mediana           12417 non-null  float64
 3   qt_alunos_avaliados               12417 non-null  int64  
 4   nu_serie                          12417 non-null  int64  
 5   qtd_fam_pobreza                   12417 non-null  int64  
 6   qtd_fam_baixa_renda               12417 non-null  int64  
 7   qtd_fam_ate_meio_sm               12417 non-null  int64  
 8   qtd_fam_renda_zero                12417 non-null  int64  
 9   taxa_atualizacao_geral_pct        12417 non-null  float64
 10  taxa_atualizacao_ate_meio_sm_pct  12417 non-null  float64
 11  qtd_escolas                       12417 non-null  int64  
 12  pct_urbana     

Pode-se observar que o número de linhas da tabela `wide_analitica_alfabetizacao` foi preservado, e as colunas não possuem valores nulos, indicando que a junção das bases de dados foi bem sucedida.

### Exportação dos dados em formato .csv para utilização no treinamento dos modelos

In [79]:
merged_wide_cad_censo.to_csv(base_path / 'features' / 'features.csv', index=False)